# 1、获取大模型

In [4]:
#导入 dotenv 库的 load_dotenv 函数，用于加载环境变量文件（.env）中的配置
import dotenv
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("DASHSCOPE_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="qwen-plus")

# 直接提供问题，并调用llm
response = llm.invoke("什么是大模型？")
print(response)

content='“大模型”是近年来人工智能领域的一个热门术语，通常指的是**大规模预训练模型**（Large Pre-trained Models），尤其是指那些参数量巨大、在海量数据上进行训练的深度学习模型。这类模型在自然语言处理（NLP）、计算机视觉（CV）、语音识别等多个领域表现出强大的能力。\n\n### 一、大模型的核心特征\n\n1. **参数量巨大**\n   - 大模型通常拥有数十亿（billion）甚至数千亿（trillion）级别的参数。\n   - 例如：GPT-3 有约 1750 亿参数，而 GPT-4 的参数量据估计更大。\n\n2. **基于Transformer架构**\n   - 绝大多数大模型采用 **Transformer** 架构（特别是自注意力机制），这使得它们能高效处理序列数据（如文本）。\n\n3. **预训练 + 微调/提示学习**\n   - 大模型通常先在大规模无标注数据上进行**预训练**（pre-training），学习通用的语言或视觉表示。\n   - 然后通过**微调**（fine-tuning）或**提示工程**（prompting）适应具体任务，如问答、翻译、摘要等。\n\n4. **涌现能力（Emergent Abilities）**\n   - 当模型规模达到一定程度时，会出现一些小模型不具备的能力，比如：\n     - 思维链推理（Chain-of-Thought Reasoning）\n     - 少样本学习（Few-shot Learning）\n     - 零样本迁移（Zero-shot Transfer）\n\n5. **多模态能力（部分大模型）**\n   - 一些大模型不仅能处理文本，还能理解图像、音频等多模态信息，如 CLIP、Flamingo、Qwen-VL 等。\n\n---\n\n### 二、典型的大模型举例\n\n| 模型 | 公司/机构 | 类型 | 参数量 |\n|------|----------|------|--------|\n| GPT-3 / GPT-3.5 / GPT-4 | OpenAI | 文本生成 | 175B+ |\n| PaLM / Gemini | Google | 文本/多模态 | 540B / 更大 |\n| LLaMA / LLaMA

# 2、使用提示词模板

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# 需要注意的一点是，这里需要指明具体的role，在这里是system和用户
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")  # {input}为变量
])

# 我们可以把prompt和具体llm的调用和在一起。
chain = prompt | llm
message = chain.invoke({"input": "大模型中的LangChain是什么?"})
print(message)

# print(type(message))

content='LangChain 是一个开源的、面向大语言模型（Large Language Model, LLM）应用开发的框架，旨在简化基于大语言模型的应用程序构建过程。它由 Harrison Chase 在 2022 年创建，迅速成为构建 LLM 驱动应用的事实标准之一。LangChain 提供了一套模块化、可组合的工具和抽象，使开发者能够高效地集成语言模型与外部数据源、工具和业务逻辑，从而构建出功能强大且智能的应用。\n\n---\n\n### 一、LangChain 的核心理念\n\nLangChain 的设计哲学是：**“语言模型不是孤立使用的，而是作为更大系统的一部分。”**  \n它强调将 LLM 与以下要素连接起来：\n\n- 外部知识库（如数据库、文档）\n- 工具（如搜索引擎、API）\n- 记忆机制（长期或短期记忆）\n- 推理与规划能力\n\n通过这种“链式”（chain）结构，LangChain 实现了复杂任务的自动化处理。\n\n---\n\n### 二、LangChain 的六大核心组件\n\n1. **Models（模型接口）**\n   - 支持多种语言模型（如 OpenAI GPT、Anthropic Claude、本地部署的 Llama 等）\n   - 提供统一的调用接口，屏蔽底层差异\n   - 包括 `LLM`（基础文本生成） 和 `ChatModel`（对话格式支持）\n\n2. **Prompts（提示工程）**\n   - 管理提示模板（Prompt Templates）\n   - 支持动态变量填充、Few-shot 示例插入\n   - 可实现提示版本管理与优化\n\n3. **Chains（链）**\n   - 将多个步骤组合成一个执行流程\n   - 例如：从文档中提取信息 → 调用 API → 生成报告\n   - 支持自定义链和预置链（如 `LLMChain`, `SequentialChain`）\n\n4. **Retrievers & Vector Stores（检索器与向量数据库）**\n   - 实现“检索增强生成”（RAG, Retrieval-Augmented Generation）\n   - 将用户问题与外部知识库匹配，提升回答准确性\n   - 支持主流向量数据库：Pineco

# 3、 使用输出解析器

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser

# 初始化模型
llm = ChatOpenAI(model="qwen-plus")

# 创建提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者。"),
    ("user", "{input}")
])

# 使用输出解析器
# output_parser = StrOutputParser()
output_parser = JsonOutputParser()

# 将其添加到上一个链中
# chain = prompt | llm
chain = prompt | llm | output_parser

# 调用它并提出同样的问题。答案是一个字符串，而不是ChatMessage
# chain.invoke({"input": "LangChain是什么?"})
chain.invoke({"input": "LangChain是什么? 用JSON格式回复，问题用question，回答用answer"})

{'question': 'LangChain是什么?',
 'answer': 'LangChain是一个开源框架，旨在简化基于大型语言模型（LLM）的应用程序开发。它提供了一套模块化工具，用于连接语言模型与外部数据源、构建上下文感知的对话系统、管理记忆、调用工具以及创建复杂的链式逻辑（chains）。LangChain支持多种集成，如向量数据库、文档加载器、API封装器等，广泛应用于聊天机器人、智能代理、自动化工作流和问答系统等场景。'}

# 4、使用向量存储

In [11]:
# 导入和使用 WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader
import bs4
import requests
r = requests.get("https://www.gov.cn/zhengce/content/202511/content_7047288.htm")
soup = bs4.BeautifulSoup(r.text, "html.parser")
## 确认定位到正确的内容
print(soup.find(id="UCAP-CONTENT"))

loader = WebBaseLoader(
        web_path="https://www.gov.cn/zhengce/content/202511/content_7047288.htm",
        bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
    )
docs = loader.load()
# print(docs)

# 对于嵌入模型，这里通过 API调用
from langchain_openai import OpenAIEmbeddings

## 在大模型构造方法的参数中加入check_embedding_ctx_length=False，以避免因上下文长度检查而导致的错误
## text-embedding-v3报错, 使用text-embedding-v2
embeddings = OpenAIEmbeddings(
    model="text-embedding-v2",
    check_embedding_ctx_length=False
)


from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(len(documents))
for i,doc in enumerate(documents):
    print(f"⭐第{i+1}条内容：")
    print(doc)
# 向量存储  embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)

<div class="b12c pages_content" id="UCAP-CONTENT">
<div class="trs_editor_view TRS_UEDITOR trs_paper_default trs_web"><p label="å±ä¸­å¯¹é½" style="margin-top: 2px; margin-bottom: 2px; text-align: center; text-indent: 0em; font-size: 24px;"><span data-index="5" style="font-size: 24px;"><strong>ä¸­åäººæ°å±åå½å½å¡é¢ä»¤â</strong></span></p><p label="å±ä¸­å¯¹é½" style="margin-top: 2px; margin-bottom: 2px; text-align: center; text-indent: 0em;"><span style="font-family: æ¥·ä½, SimKai;">ç¬¬820å·</span></p><p style="margin-top: 2px; margin-bottom: 2px; text-indent: 0em;"><br/></p><p data-index="-2" style="text-indent: 2em; margin-top: 2px; margin-bottom: 2px;">ãçæç¯å¢çæµæ¡ä¾ãå·²ç»2025å¹´10æ17æ¥å½å¡é¢ç¬¬70æ¬¡å¸¸å¡ä¼è®®éè¿ï¼ç°äºå¬å¸ï¼èª2026å¹´1æ1æ¥èµ·æ½è¡ã</p><p label="å³å¯¹é½" style="margin-top: 2px; margin-bottom: 2px; text-indent: 0em; text-align: right;">æ»çããæå¼ºãããããããâ</p><p label="å³å¯¹é½" style="marg

# 5、RAG(检索增强生成)

In [12]:
from langchain_core.prompts import PromptTemplate

## 上面得到的向量数据库vector
retriever = vector.as_retriever()
## 取前三条数据
retriever.search_kwargs = {"k": 3}
## 向量数据库根据问题检索到的内部文档
docs = retriever.invoke("生态环境监测条例是什么？")

# for i,doc in enumerate(docs):
#     print(f"⭐第{i+1}条规定：")
#     print(doc)

# 6.定义提示词模版
prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息:
{info}

用户问：
{question}

请用中文回答用户问题。
"""
# 7.得到提示词模版对象
template = PromptTemplate.from_template(prompt_template)

# 8.得到提示词对象
prompt = template.format(info=docs, question='生态环境监测条例是什么？')

## 9. 调用LLM
response = llm.invoke(prompt)
print(response.content)

《生态环境监测条例》是中华人民共和国国务院令第820号，已经2025年10月17日国务院第70次常务会议通过，自2026年1月1日起施行。该条例旨在规范生态环境监测活动，提升监测能力和水平，保障监测数据质量，更好发挥生态环境监测在支撑生态文明和美丽中国建设、服务经济社会高质量发展中的重要作用。

条例适用于在中华人民共和国领域及管辖的其他海域开展的生态环境监测及其相关活动。所称生态环境监测包括政府及其有关部门为履行生态环境保护职责开展的公共监测，以及企事业单位等负有法定监测义务的主体对其活动影响环境所开展的自行监测。

条例明确坚持依法监测、科学监测、诚信监测的原则，构建政府主导、部门协同、企事业单位履责、社会参与、公众监督的工作机制。国家建立健全生态环境监测制度，完善监测规范和标准，并加强监测能力建设，推动建立陆海统筹、天地一体、上下协同、信息共享的现代化生态环境监测网络。

县级以上人民政府应加强对监测工作的组织领导，将所需经费纳入本级财政预算。国务院生态环境主管部门负责全国生态环境监测工作的监督管理，其他相关部门按职责分工负责有关监测工作；地方各级政府相应主管部门在其行政区域内履行监管职责。

此外，条例强化了对监测数据弄虚作假行为的惩处机制：对篡改、伪造监测数据的单位和个人设定严厉处罚，包括罚款、停产停业、禁止从事监测服务乃至追究刑事责任。技术服务机构弄虚作假的，最高可处200万元罚款，并吊销资质证书；相关责任人员也将面临罚款及从业禁止，构成犯罪的终身禁止从业。

违反条例造成他人损失的，依法承担赔偿责任；违反治安管理或构成犯罪的，依法追究相应法律责任。军队的生态环境监测工作按照中央军事委员会有关规定执行，生态环境监测数据的安全保护依照《中华人民共和国数据安全法》和《网络数据安全管理条例》执行。


# 6、使用Agent

In [22]:
from langchain.tools.retriever import create_retriever_tool
# 检索器工具
retriever_tool = create_retriever_tool(
    retriever,
    "CivilCodeRetriever",
    "搜索有关中华人民共和国民法典的信息。关于中华人民共和国民法典的任何问题，您必须使用此工具!",
)

tools = [retriever_tool]

from langchain import hub
from langchain.agents import create_openai_functions_agent

from langchain.agents import AgentExecutor

# https://smith.langchain.com/hub
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)

# 运行代理
agent_executor.invoke({"input":"生态环境监测条例是什么"})

e:\github\langchain-practice\.venv\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...
您提到的“生态环境监测条例”并不属于《中华人民共和国民法典》的内容，因此我无法通过民法典检索工具为您提供相关信息。该条例可能属于环境保护或行政法规范畴。

建议您查阅《中华人民共和国环境保护法》或相关生态环境部门发布的专门规定。如果需要，我可以帮助您进一步查找其他信息。

> Finished chain.


{'input': '生态环境监测条例是什么',
 'output': '您提到的“生态环境监测条例”并不属于《中华人民共和国民法典》的内容，因此我无法通过民法典检索工具为您提供相关信息。该条例可能属于环境保护或行政法规范畴。\n\n建议您查阅《中华人民共和国环境保护法》或相关生态环境部门发布的专门规定。如果需要，我可以帮助您进一步查找其他信息。'}